# 05.3 - RAG Hybrid + QR + CR dengan Llama 3.2 3B (Ollama Lokal)

Notebook ini mengevaluasi konfigurasi **Hybrid Retrieval** dengan Llama 3.2 3B (Ollama Lokal) sebagai
model generasi (via Ollama lokal API), menggabungkan BM25 (sparse) dan Dense retrieval
(Ollama `nomic-embed-text`) menggunakan Reciprocal Rank Fusion (RRF).

**Pipeline:**
```
original query
  -> QR rewrite (Llama 3.2 3B (Ollama Lokal) via Ollama lokal)
  -> BM25 top-50 + Dense top-50 (Ollama nomic-embed-text)
  -> RRF fusion -> top-20 kandidat
  -> CrossEncoder rerank
  -> top-5
  -> Llama 3.2 3B (Ollama Lokal) generate (pakai ORIGINAL query)
```

**Model:**
- Generator: `meta-llama/meta-llama/llama3.2` via Ollama lokal API
- Embedder: `nomic-embed-text` (Ollama local, 768 dim, gratis)
- Vector DB: chromadb persistent (path: `pubmedqa_chroma_llama/`)

**Evaluator:** Custom zero-NaN 4 metrik (faithfulness, context_recall, answer_relevancy, context_precision)

**Prerequisites:**
1. Ollama service running (`ollama serve`) dengan model `nomic-embed-text` terpasang
2. OPENROUTER_API_KEY environment variable terset
3. Install: `pip install openai ollama rank-bm25 chromadb datasets`

In [1]:
# Install dependencies (jalankan sekali saja)
# !pip install openai ollama rank-bm25 chromadb datasets

In [2]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime

from openai import OpenAI
from rank_bm25 import BM25Okapi
from datasets import load_dataset
import chromadb
from sentence_transformers import CrossEncoder

warnings.filterwarnings('ignore')
print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | chromadb: {chromadb.__version__}')

C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Semua library berhasil diimpor!
Python: 3.11.9 | NumPy: 2.3.5 | chromadb: 1.5.8


In [3]:
# ============================================================
# KONFIGURASI
# ============================================================
LLM_MODEL   = 'llama3.2'  # Llama 3.2 3B lokal via Ollama
EMBED_MODEL = 'nomic-embed-text'  # Ollama local, 768 dim, gratis

TOP_K_BM25       = 50   # BM25 ambil top-50
TOP_K_DENSE      = 50   # Dense ambil top-50
TOP_K_CANDIDATES = 20   # RRF ambil top-20 kandidat
RERANKER_MODEL   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
TOP_K_RETRIEVAL  = 5    # Final top-5 setelah CrossEncoder

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'
MAX_SAMPLES    = 500

TEMPERATURE = 0.0
SEED        = 42

NOTEBOOK_DIR    = Path('.')
BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'
CHROMA_DB_PATH  = NOTEBOOK_DIR / 'pubmedqa_chroma_llama'  # nomic-embed-text (768 dim)
RESULTS_DIR     = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

CONFIG_NAME        = 'qr_cr_llama32'
PHASE1_PATH        = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'
PHASE2_CUSTOM_PATH = RESULTS_DIR / f'{CONFIG_NAME}_phase2_custom.json'

print('Konfigurasi:')
print(f'  LLM          : {LLM_MODEL} (via Ollama lokal)')
print(f'  Embedder     : {EMBED_MODEL}')
print(f'  Retriever    : BM25 top-{TOP_K_BM25} + Dense top-{TOP_K_DENSE} -> RRF -> top-{TOP_K_RETRIEVAL}')
print(f'  Vector DB    : chromadb (path: {CHROMA_DB_PATH})')
print(f'  Sampel       : {MAX_SAMPLES}')
print(f'  Config       : {CONFIG_NAME}')
print()


Konfigurasi:
  LLM          : llama3.2 (via Ollama lokal)
  Embedder     : nomic-embed-text
  Retriever    : BM25 top-50 + Dense top-50 -> RRF -> top-5
  Vector DB    : chromadb (path: pubmedqa_chroma_llama)
  Sampel       : 500
  Config       : qr_cr_llama32



In [4]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str


@dataclass
class RetrievalResult:
    document       : Document
    score          : float           # BM25 score (atau RRF score)
    doc_id         : int = -1        # index di documents list
    bm25_score     : float = 0.0
    dense_score    : float = 0.0
    rrf_score      : float = 0.0
    reranker_score : float = 0.0


def tokenize_bm25(text: str) -> List[str]:
    """Tokenizer untuk BM25: hapus tanda baca, lowercase, split spasi."""
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()


sample_text = 'Does aspirin (75mg) reduce myocardial infarction risk?'
print(f'Tokenisasi BM25: {tokenize_bm25(sample_text)}')
print('Data classes dan tokenizer siap.')

Tokenisasi BM25: ['does', 'aspirin', '75mg', 'reduce', 'myocardial', 'infarction', 'risk']
Data classes dan tokenizer siap.


In [5]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    print(f'Total potongan dokumen: {len(docs)}')
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index baru...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        print(f'Index disimpan ke {BM25_INDEX_PATH}')
        return bm25, documents


_full_data            = load_dataset(DATASET_NAME, DATASET_SUBSET, trust_remote_code=True)['train']
bm25_index, documents = load_or_build_bm25(_full_data.select(range(500)))
pubmedqa_data         = _full_data.select(range(MAX_SAMPLES))
print(f'\nEvaluasi akan menggunakan {len(pubmedqa_data)} sampel pertama.')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memuat BM25 index dari pubmedqa_bm25.pkl...
Dimuat: 1706 dokumen

Evaluasi akan menggunakan 500 sampel pertama.


In [6]:
# ============================================================
# Setup Ollama Client (LLM Generator + Embeddings)
# ============================================================
# Semua via Ollama lokal: tanpa biaya, tanpa rate limit, tanpa internet.
# Pastikan Ollama service jalan: `ollama serve`
# Pastikan model sudah di-pull:
#   ollama pull llama3.2          (model generator, ~2 GB)
#   ollama pull nomic-embed-text  (model embedding, ~270 MB)

import ollama

# Cek Ollama running dan model tersedia
try:
    _models = ollama.list()
    _names  = [m.get('model', m.get('name', '')) for m in _models.get('models', [])]
    print(f'Ollama siap. {len(_names)} model tersedia.')
    # Soft-check model yang dibutuhkan
    needed = [LLM_MODEL, EMBED_MODEL]
    for m in needed:
        ok = any(m in n for n in _names)
        mark = 'OK' if ok else 'MISSING'
        print(f'  [{mark}] {m}')
        if not ok:
            print(f'         Jalankan: ollama pull {m}')
except Exception as e:
    print(f'ERROR: Ollama tidak jalan: {e}')
    print('Pastikan Ollama service aktif: `ollama serve`')
    raise


def llama_generate(prompt: str, max_tokens: int = 300, temperature: float = TEMPERATURE) -> str:
    """Wrapper Ollama chat API (Llama 3.2 3B lokal) dengan retry ringan."""
    for attempt in range(3):
        try:
            response = ollama.chat(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                options={
                    'temperature' : temperature,
                    'num_predict' : max_tokens,
                    'seed'        : SEED,
                },
            )
            return response['message']['content'].strip()
        except Exception as e:
            if attempt == 2:
                print(f'  [Ollama Error final] {type(e).__name__}: {str(e)[:100]}')
                raise
            print(f'  [Ollama Error] {type(e).__name__}: {str(e)[:80]} -- retry {attempt+1}/3')
            time.sleep(3)
    raise RuntimeError('Ollama gagal setelah 3 percobaan.')


def ollama_embed(texts: List[str], model: str = EMBED_MODEL) -> List[List[float]]:
    """Wrapper Ollama embeddings (nomic-embed-text, 768 dim, lokal)."""
    embeddings = []
    for text in texts:
        for attempt in range(3):
            try:
                resp = ollama.embeddings(model=model, prompt=text)
                embeddings.append(resp['embedding'])
                break
            except Exception as e:
                if attempt == 2:
                    raise
                time.sleep(2)
    return embeddings


# Smoke test
print('\nTesting Ollama LLM (Llama 3.2)...')
_test = llama_generate('Reply with exactly: OK', max_tokens=10)
print(f'  Response: {_test!r}')

print('\nTesting Ollama embeddings (nomic-embed-text)...')
_emb = ollama_embed(['aspirin reduces heart attack risk'])
print(f'  Embed dim: {len(_emb[0])} (expected 768 untuk nomic-embed-text)')

print('\nSemua client siap (Ollama lokal):')
print(f'  Generator : {LLM_MODEL}')
print(f'  Embedder  : {EMBED_MODEL}')


Ollama siap. 2 model tersedia.
  [OK] llama3.2
  [OK] nomic-embed-text

Testing Ollama LLM (Llama 3.2)...
  Response: 'OK'

Testing Ollama embeddings (nomic-embed-text)...
  Embed dim: 768 (expected 768 untuk nomic-embed-text)

Semua client siap (Ollama lokal):
  Generator : llama3.2
  Embedder  : nomic-embed-text


In [7]:
# ============================================================
# Build / Load ChromaDB index dengan Ollama embeddings (nomic-embed-text)
# Embed sekali, di-cache di CHROMA_DB_PATH
# ============================================================

def build_or_load_chroma_index(documents: List[Document]) -> chromadb.Collection:
    """
    Build ChromaDB persistent collection dengan Ollama embeddings (nomic-embed-text).
    Kalau sudah ada dan count-nya match, load saja.
    """
    chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
    collection_name = 'pubmedqa_docs'

    try:
        collection = chroma_client.get_collection(name=collection_name)
        count = collection.count()
        if count == len(documents):
            print(f'Load Chroma collection "{collection_name}" ({count} dokumen) dari {CHROMA_DB_PATH}')
            return collection
        else:
            print(f'Count mismatch: chroma={count}, documents={len(documents)}. Rebuild...')
            chroma_client.delete_collection(name=collection_name)
    except Exception:
        pass

    print(f'Build Chroma collection ({len(documents)} dokumen)...')
    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={'hnsw:space': 'cosine'}
    )

    # Batch embedding
    BATCH = 100
    t0 = time.time()
    for start in range(0, len(documents), BATCH):
        batch_docs = documents[start:start + BATCH]
        batch_texts = [d.text[:8000] for d in batch_docs]  # truncate untuk safety
        batch_ids   = [str(start + i) for i in range(len(batch_docs))]
        batch_meta  = [{'pubid': d.pubid, 'section': d.section_label} for d in batch_docs]

        embeddings = ollama_embed(batch_texts)
        collection.add(
            ids=batch_ids,
            documents=batch_texts,
            metadatas=batch_meta,
            embeddings=embeddings,
        )
        done = start + len(batch_docs)
        eta  = (time.time()-t0)/done*(len(documents)-done)/60 if done < len(documents) else 0
        print(f'  [{done}/{len(documents)}] embedded | ETA {eta:.1f} mnt')

    print(f'Chroma index dibangun dalam {(time.time()-t0)/60:.1f} menit')
    return collection


chroma_collection = build_or_load_chroma_index(documents)
print(f'\nChroma index siap: {chroma_collection.count()} dokumen')

Load Chroma collection "pubmedqa_docs" (1706 dokumen) dari pubmedqa_chroma_llama

Chroma index siap: 1706 dokumen


In [8]:
# ============================================================
# Load CrossEncoder Reranker
# ============================================================

print(f'Loading CrossEncoder: {RERANKER_MODEL}...')
t0 = time.time()
cross_encoder = CrossEncoder(RERANKER_MODEL)
print(f'CrossEncoder siap dalam {time.time()-t0:.1f} detik')

# Smoke test
_pairs = [
    ('Does aspirin prevent heart attacks?', 'Aspirin reduces platelet aggregation.'),
    ('Does aspirin prevent heart attacks?', 'Weather patterns affect agriculture.'),
]
_scores = cross_encoder.predict(_pairs)
print(f'Smoke test: relevan={_scores[0]:.3f}, tidak relevan={_scores[1]:.3f}')
assert _scores[0] > _scores[1], 'CrossEncoder gagal membedakan relevan vs tidak!'
print('Reranker berfungsi.')

Loading CrossEncoder: cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights: 100%|███████████████████████| 105/105 [00:00<00:00, 405.18it/s, Materializing param=classifier.weight]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CrossEncoder siap dalam 4.7 detik
Smoke test: relevan=-0.520, tidak relevan=-11.158
Reranker berfungsi.


In [9]:
# ============================================================
# Query Rewriting
# ============================================================

QUERY_REWRITE_PROMPT = (
    'You are a query rewriting assistant for a biomedical question-answering system.\n'
    'Rewrite the following medical question to improve retrieval from a PubMed research database.\n\n'
    'Rules:\n'
    '1. Be more specific and add relevant medical/scientific terminology.\n'
    '2. Expand abbreviations (e.g. "MI" -> "myocardial infarction").\n'
    '3. Preserve the original yes/no/maybe answerable intent.\n'
    '4. Output ONLY the rewritten question, no explanations.\n\n'
    'Original question: {query}\n\n'
    'Rewritten question:'
)


def rewrite_query(query: str) -> str:
    """Reformulasi query untuk meningkatkan kualitas retrieval."""
    prompt = QUERY_REWRITE_PROMPT.format(query=query)
    try:
        rewritten = llama_generate(prompt, max_tokens=150, temperature=0.3)
        rewritten = rewritten.strip().replace('\n', ' ')
        return rewritten if len(rewritten) >= 10 else query
    except Exception as e:
        print(f'  [QR Error] {e} -- pakai query asli')
        return query


# Test QR
print('Contoh Query Rewriting:')
for q in ['Does aspirin reduce the risk of MI?',
          'Can exercise prevent T2DM?']:
    rw = rewrite_query(q)
    print(f'\nAsli    : {q}')
    print(f'Rewrite : {rw}')

Contoh Query Rewriting:

Asli    : Does aspirin reduce the risk of MI?
Rewrite : Does low-dose aspirin therapy significantly reduce the incidence of recurrent myocardial infarction in patients with established coronary artery disease?

Asli    : Can exercise prevent T2DM?
Rewrite : Can regular aerobic exercise interventions effectively reduce or prevent the onset of type 2 diabetes mellitus in adults?


In [10]:
# ============================================================
# Dense retrieval + RRF fusion
# ============================================================

def retrieve_dense(query: str, k: int = 50) -> List[Tuple[int, float]]:
    """
    Dense retrieval via ChromaDB.
    Returns: list of (doc_id, distance_score) sorted by relevance.
    """
    qvec = ollama_embed([query])[0]
    results = chroma_collection.query(
        query_embeddings=[qvec],
        n_results=k,
        include=['distances']
    )
    doc_ids   = [int(i) for i in results['ids'][0]]
    distances = results['distances'][0]  # cosine distance (lower = better)
    # Convert to similarity score (1 - distance) for consistency
    scores = [1.0 - d for d in distances]
    return list(zip(doc_ids, scores))


def retrieve_bm25_raw(query: str, k: int = 50) -> List[Tuple[int, float]]:
    """BM25 retrieval, return (doc_id, bm25_score) sorted desc."""
    tokens = tokenize_bm25(query)
    scores = bm25_index.get_scores(tokens)
    top_k  = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_k]


def reciprocal_rank_fusion(
    rank_lists: List[List[int]],
    k: int = 60
) -> List[Tuple[int, float]]:
    """
    Reciprocal Rank Fusion.
    rank_lists: list of rank lists, each berisi doc_ids terurut.
    k: konstanta RRF (default 60 dari literatur).
    Returns: list of (doc_id, rrf_score) sorted desc.
    """
    scores = {}
    for rank_list in rank_lists:
        for rank, doc_id in enumerate(rank_list):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])


def retrieve_hybrid(
    query: str,
    k_bm25: int = TOP_K_BM25,
    k_dense: int = TOP_K_DENSE,
    k_candidates: int = TOP_K_CANDIDATES,
    k_final: int = TOP_K_RETRIEVAL
) -> List[RetrievalResult]:
    """
    Hybrid retrieval: BM25 top-k_bm25 + Dense top-k_dense, merged via RRF.
    Return top-k_final by RRF score.
    """
    # Query rewriting
    rewritten = rewrite_query(query)

    bm25_results  = retrieve_bm25_raw(rewritten, k=k_bm25)
    dense_results = retrieve_dense(rewritten, k=k_dense)

    # Bangun lookup untuk skor asli
    bm25_scores  = dict(bm25_results)
    dense_scores = dict(dense_results)

    # RRF fusion
    rank_lists = [
        [doc_id for doc_id, _ in bm25_results],
        [doc_id for doc_id, _ in dense_results],
    ]
    fused = reciprocal_rank_fusion(rank_lists, k=60)

    # Build candidates (top-K from RRF)
    results = []
    for doc_id, rrf_score in fused[:k_candidates]:
        results.append(RetrievalResult(
            document=documents[doc_id],
            score=rrf_score,  # score utama = RRF
            doc_id=doc_id,
            bm25_score=bm25_scores.get(doc_id, 0.0),
            dense_score=dense_scores.get(doc_id, 0.0),
            rrf_score=rrf_score,
        ))

    # CrossEncoder reranking
    pairs = [(query, r.document.text) for r in results]
    ce_scores = cross_encoder.predict(pairs)
    for r, s in zip(results, ce_scores):
        r.reranker_score = float(s)
    results = sorted(results, key=lambda r: r.reranker_score, reverse=True)[:k_final]

    return results, rewritten


# Test retrieval
test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r, test_rw = retrieve_hybrid(test_q)
print(f'Query asli : {test_q}')
print(f'Rewrite    : {test_rw}')
print(f'\nTop-{TOP_K_RETRIEVAL} dokumen (Hybrid BM25+Dense via RRF):')
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] CE={r.reranker_score:.3f} | RRF={r.rrf_score:.4f} | BM25={r.bm25_score:.2f} | Dense={r.dense_score:.3f} '
          f'| {r.document.section_label} | {r.document.text[:70]}...')

Query asli : Does aspirin reduce the risk of myocardial infarction?
Rewrite    : Does low-dose aspirin therapy significantly decrease the incidence of recurrent myocardial infarction in patients with established coronary artery disease?

Top-5 dokumen (Hybrid BM25+Dense via RRF):
  [1] CE=-2.540 | RRF=0.0161 | BM25=0.00 | Dense=0.695 | BACKGROUND AND PURPOSE | In primary and secondary prevention trials, statins have been shown to...
  [2] CE=-3.116 | RRF=0.0299 | BM25=20.12 | Dense=0.683 | OBJECTIVE | Myocardial damage that is associated with percutaneous coronary interv...
  [3] CE=-4.325 | RRF=0.0232 | BM25=16.57 | Dense=0.620 | MEASUREMENTS AND RESULTS | Patients who underwent either cold (26 degrees +/- 5 degrees C) or war...
  [4] CE=-5.150 | RRF=0.0301 | BM25=27.60 | Dense=0.655 | DESIGN | Within a prospective, population-based cohort study individuals withou...
  [5] CE=-5.379 | RRF=0.0291 | BM25=17.85 | Dense=0.685 | BACKGROUND | It has recently been shown that non-high density

In [11]:
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Carefully read the context and assess whether it supports or refutes the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n'
    '  - no    : the evidence refutes or does not support the hypothesis\n'
    '  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n'
    '            others say no), or if the context contains no relevant information at all\n'
    '- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n'
    '  Do NOT use maybe simply because the evidence is limited or not 100%% certain.\n\n'
    'Answer:'
)


def generate_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    """Generate jawaban via OpenRouter pakai original query."""
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    return llama_generate(
        GENERATION_PROMPT.format(context=context, question=query),
        max_tokens=300,
        temperature=TEMPERATURE
    )


# Test
test_ans = generate_answer(test_q, test_r)
print('Output generation:')
print('-' * 60)
print(test_ans)
print('-' * 60)

Output generation:
------------------------------------------------------------
The provided abstracts do not directly address the question of whether aspirin reduces the risk of myocardial infarction. However, they do mention that statins have been shown to reduce the risk of stroke and have antiatherothrombotic properties, which may imply a potential benefit of antiplatelet agents like aspirin in reducing cardiovascular events.

The context suggests that aspirin is often used as an adjunct therapy to statins for secondary prevention of cardiovascular events. While the abstracts do not provide direct evidence on the effectiveness of aspirin, they do support the idea that certain lipid-lowering and anti-inflammatory therapies may have a beneficial effect on cardiovascular outcomes.

Therefore, based on the indirect information provided in the context, I would answer:

yes
------------------------------------------------------------


In [12]:
def extract_label(answer: str) -> str:
    """Ekstrak prediksi yes/no/maybe dari teks jawaban."""
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'


cases = [
    ('Strong evidence.\nyes', 'yes'),
    ('No effect found.\nno',  'no'),
    ('Mixed results.\nmaybe', 'maybe'),
    ('Verdict: yes.',          'yes'),
    ('Totally unclear.',       'maybe'),
]
all_ok = all(extract_label(txt) == exp for txt, exp in cases)
print(f'Unit test extract_label: {"PASS" if all_ok else "FAIL"}')
print(f'Label dari test answer: {extract_label(test_ans)!r}')

Unit test extract_label: PASS
Label dari test answer: 'yes'


In [13]:
# ============================================================
# Custom Zero-NaN Evaluator — 4 metrik via Ollama lokal
# ============================================================

def _split_sentences(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]


def _llm_yes_no(prompt: str) -> bool:
    try:
        resp = llama_generate(prompt, max_tokens=10, temperature=0.0)
        return 'yes' in resp.lower()[:15]
    except Exception:
        return False


def compute_faithfulness(answer: str, contexts: List[str]) -> float:
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement directly supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    supported = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s)))
    return supported / len(sentences)


def compute_context_recall(reference: str, contexts: List[str]) -> float:
    sentences = _split_sentences(reference)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    covered = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s)))
    return covered / len(sentences)


def compute_answer_relevancy(question: str, answer: str) -> float:
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    prompt_tmpl = (
        'Question: {question}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement relevant to answering the question above? '
        'Answer with only "yes" or "no".'
    )
    relevant = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(question=question, sent=s)))
    return relevant / len(sentences)


def compute_context_precision(question: str, contexts: List[str], reference: str) -> float:
    if not contexts:
        return 0.0
    prompt_tmpl = (
        'Question: {question}\n\n'
        'Ground truth answer: {reference}\n\n'
        'Retrieved context: {ctx}\n\n'
        'Does this context contain information useful for correctly answering '
        'the question based on the ground truth? Answer with only "yes" or "no".'
    )
    relevance = []
    for ctx in contexts:
        is_rel = _llm_yes_no(prompt_tmpl.format(
            question=question, reference=reference[:300], ctx=ctx[:400]
        ))
        relevance.append(1 if is_rel else 0)
    total_relevant = sum(relevance)
    if total_relevant == 0:
        return 0.0
    precision_sum = 0.0
    relevant_count = 0
    for k, rel in enumerate(relevance):
        if rel:
            relevant_count += 1
            precision_sum += relevant_count / (k + 1)
    return precision_sum / total_relevant


def evaluate_custom(question: str, answer: str,
                    contexts: List[str], reference: str) -> Dict:
    return {
        'faithfulness'      : compute_faithfulness(answer, contexts),
        'context_recall'    : compute_context_recall(reference, contexts),
        'answer_relevancy'  : compute_answer_relevancy(question, answer),
        'context_precision' : compute_context_precision(question, contexts, reference),
    }


# Smoke test
_ctx = ['Aspirin reduces blood clotting and is used for heart attack prevention.']
_ans = 'Aspirin helps prevent heart attacks. It works by reducing clotting.'
_ref = 'Aspirin is used for heart attack prevention by reducing blood clotting.'
_r   = evaluate_custom('Does aspirin prevent heart attacks?', _ans, _ctx, _ref)
print('Smoke test (4 metrik):')
for k, v in _r.items():
    print(f'  {k} = {v:.3f}')
print('Zero-NaN evaluator siap.')

Smoke test (4 metrik):
  faithfulness = 1.000
  context_recall = 1.000
  answer_relevancy = 1.000
  context_precision = 1.000
Zero-NaN evaluator siap.


## Demo — 5 Sampel Pertama

In [ ]:
DEMO_SIZE = 5

print(f'DEMO: {DEMO_SIZE} sampel pertama (Hybrid BM25+Dense via RRF, {LLM_MODEL})')
print('=' * 65)

for i in range(DEMO_SIZE):
    s  = pubmedqa_data[i]
    q  = s['question']
    gt = s['final_decision']

    retrieved = retrieve_hybrid(q)
    answer    = generate_answer(q, retrieved)
    predicted = extract_label(answer)
    correct   = predicted == gt

    verdict = 'BENAR' if correct else 'SALAH'
    print(f'\n[{i+1}/{DEMO_SIZE}] {q[:75]}...')
    print(f'  Top-1 RRF={retrieved[0].rrf_score:.4f} (BM25={retrieved[0].bm25_score:.2f}, Dense={retrieved[0].dense_score:.3f})')
    print(f'  GT={gt} | Pred={predicted} | {verdict}')
    print(f'  Jawaban: {answer[:120]}...')

## Phase 1 — Generate Jawaban (500 Sampel)

Estimasi waktu: ~20-30 menit dengan Llama 3.3 70B via OpenRouter. Resume otomatis.

In [14]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Memulai Fase 1: {MAX_SAMPLES} sampel (Hybrid BM25+Dense).')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()

    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']

        retrieved, rewritten = retrieve_hybrid(q)
        answer    = generate_answer(q, retrieved)
        predicted = extract_label(answer)

        phase1_results.append({
            'idx'             : i,
            'pubid'           : str(s['pubid']),
            'question'        : q,
            'rewritten_query' : rewritten,
            'ground_truth'    : gt,
            'predicted_label' : predicted,
            'is_correct'      : predicted == gt,
            'answer'          : answer,
            'contexts'        : [r.document.text for r in retrieved],
            'reference'       : ref,
            'retrieval_scores': [r.bm25_score  for r in retrieved],
            'dense_scores'    : [r.dense_score for r in retrieved],
            'rrf_scores'      : [r.rrf_score   for r in retrieved],
            'reranker_scores' : [r.reranker_score for r in retrieved],
        })

        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({'config': CONFIG_NAME,
                           'llm_model': LLM_MODEL,
                           'embed_model': EMBED_MODEL,
                           'timestamp': datetime.now().isoformat(),
                           'max_samples': MAX_SAMPLES, 'completed': i+1,
                           'results': phase1_results}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Akurasi: {acc:.1%} | pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')

    print(f'\nFase 1 selesai! -> {PHASE1_PATH}')
else:
    print(f'Fase 1 sudah selesai ({MAX_SAMPLES} sampel).')

Memulai Fase 1: 500 sampel (Hybrid BM25+Dense).
Memproses 500 sampel tersisa...

  [ 10/500] Akurasi: 50.0% | pred=yes, gt=yes | ETA 574.7 mnt
  [ 20/500] Akurasi: 60.0% | pred=yes, gt=yes | ETA 533.6 mnt
  [ 30/500] Akurasi: 63.3% | pred=yes, gt=yes | ETA 516.6 mnt
  [ 40/500] Akurasi: 55.0% | pred=yes, gt=no | ETA 503.6 mnt
  [ 50/500] Akurasi: 58.0% | pred=yes, gt=no | ETA 497.5 mnt
  [ 60/500] Akurasi: 56.7% | pred=yes, gt=yes | ETA 479.4 mnt
  [ 70/500] Akurasi: 55.7% | pred=yes, gt=yes | ETA 474.5 mnt
  [ 80/500] Akurasi: 57.5% | pred=yes, gt=yes | ETA 495.8 mnt
  [ 90/500] Akurasi: 56.7% | pred=yes, gt=maybe | ETA 476.7 mnt
  [100/500] Akurasi: 58.0% | pred=yes, gt=yes | ETA 464.8 mnt
  [110/500] Akurasi: 56.4% | pred=yes, gt=yes | ETA 449.0 mnt
  [120/500] Akurasi: 55.0% | pred=yes, gt=yes | ETA 432.7 mnt
  [130/500] Akurasi: 54.6% | pred=yes, gt=maybe | ETA 416.9 mnt
  [140/500] Akurasi: 55.7% | pred=yes, gt=yes | ETA 402.8 mnt
  [150/500] Akurasi: 55.3% | pred=yes, gt=no | ET

## Analisis Phase 1

In [15]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']

n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)

print(f'ANALISIS PHASE 1 - {n} sampel ({CONFIG_NAME})')
print('=' * 55)
print(f'Label Accuracy    : {n_correct}/{n} = {n_correct/n:.1%}')
print(f'Hallucination Rate: {(n-n_correct)/n:.1%}\n')

# Per-label
print('Per-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = [r for r in results_p1 if r['ground_truth'] == lbl]
    if sub:
        c = sum(r['is_correct'] for r in sub)
        print(f'  {lbl:>5}: {c}/{len(sub)} = {c/len(sub):.1%}')

# Prediction distribution
preds = [r['predicted_label'] for r in results_p1]
print(f'\nDistribusi prediksi:')
for lbl in ['yes','no','maybe']:
    print(f'  {lbl:>5}: {preds.count(lbl)} ({preds.count(lbl)/n:.0%})')

# Hybrid-specific: overlap BM25 vs Dense
overlap_ratios = []
for r in results_p1:
    # top-5 doc ids would need to be re-computed; instead compare scores
    # cek apakah rank tertinggi BM25 dan Dense overlap di top-5 final
    pass

print(f'\nRata-rata skor BM25  : {np.mean([np.mean(r["retrieval_scores"]) for r in results_p1]):.4f}')
print(f'Rata-rata skor Dense : {np.mean([np.mean(r["dense_scores"])     for r in results_p1]):.4f}')
print(f'Rata-rata skor RRF   : {np.mean([np.mean(r["rrf_scores"])       for r in results_p1]):.4f}')

ANALISIS PHASE 1 - 500 sampel (qr_cr_llama32)
Label Accuracy    : 275/500 = 55.0%
Hallucination Rate: 45.0%

Per-label accuracy:
    yes: 250/275 = 90.9%
     no: 23/159 = 14.5%
  maybe: 2/66 = 3.0%

Distribusi prediksi:
    yes: 437 (87%)
     no: 49 (10%)
  maybe: 14 (3%)

Rata-rata skor BM25  : 25.6726
Rata-rata skor Dense : 0.6821
Rata-rata skor RRF   : 0.0273


## Phase 2 — Custom Evaluator 4 Metrik (500 Sampel)

Estimasi waktu: ~20-30 menit.

In [16]:
MAX_CUSTOM_SAMPLES = 500
REQUIRED_METRICS = ['faithfulness', 'context_recall', 'answer_relevancy', 'context_precision']

with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_custom = json.load(f)['results'][:MAX_CUSTOM_SAMPLES]

if PHASE2_CUSTOM_PATH.exists():
    with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
        p2_custom = json.load(f)['results']
    done_custom = {r['idx'] for r in p2_custom if all(m in r for m in REQUIRED_METRICS)}
    needs_upgrade = [r for r in p2_custom if not all(m in r for m in REQUIRED_METRICS)]
    print(f'Resume: {len(done_custom)}/{MAX_CUSTOM_SAMPLES} selesai dengan 4 metrik.')
    if needs_upgrade:
        print(f'Perlu upgrade: {len(needs_upgrade)} sampel.')
else:
    p2_custom, done_custom, needs_upgrade = [], set(), []
    print(f'Mulai: {MAX_CUSTOM_SAMPLES} sampel (custom zero-NaN, 4 metrik).')

# Tahap 1: Upgrade
if needs_upgrade:
    print(f'\nTahap 1: Upgrade {len(needs_upgrade)} sampel...')
    t_up = time.time()
    p1_lookup = {r['idx']: r for r in p1_custom}
    for i, r in enumerate(needs_upgrade):
        src = p1_lookup[r['idx']]
        if 'answer_relevancy' not in r:
            r['answer_relevancy'] = compute_answer_relevancy(src['question'], src['answer'])
        if 'context_precision' not in r:
            r['context_precision'] = compute_context_precision(src['question'], src['contexts'], src['reference'])
        done_custom.add(r['idx'])
        if (i + 1) % 5 == 0 or i == len(needs_upgrade) - 1:
            with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
                json.dump({
                    'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                    'max_samples': MAX_CUSTOM_SAMPLES,
                    'metrics': REQUIRED_METRICS,
                    'evaluator': 'custom_zero_nan_4metrics',
                    'results': p2_custom
                }, f, indent=2, ensure_ascii=False)
            done  = i + 1
            eta   = (time.time()-t_up)/done*(len(needs_upgrade)-done)/60 if done < len(needs_upgrade) else 0
            print(f'  upgrade [{done:3d}/{len(needs_upgrade)}] | ETA {eta:.1f} mnt')

# Tahap 2: Evaluasi sampel baru
remaining = [r for r in p1_custom if r['idx'] not in done_custom]
print(f'\nTahap 2: Evaluasi {len(remaining)} sampel baru...\n')

t0 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_custom(r['question'], r['answer'], r['contexts'], r['reference'])
    p2_custom.append({
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        **scores
    })
    if (i + 1) % 5 == 0 or i == len(remaining) - 1:
        with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
            json.dump({
                'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                'max_samples': MAX_CUSTOM_SAMPLES,
                'metrics': REQUIRED_METRICS,
                'evaluator': 'custom_zero_nan_4metrics',
                'results': p2_custom
            }, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining)
        eta   = (time.time()-t0)/done*(total-done)/60 if done < total else 0
        avg_f  = sum(x['faithfulness']      for x in p2_custom) / len(p2_custom)
        avg_cr = sum(x['context_recall']    for x in p2_custom) / len(p2_custom)
        avg_ar = sum(x['answer_relevancy']  for x in p2_custom) / len(p2_custom)
        avg_cp = sum(x['context_precision'] for x in p2_custom) / len(p2_custom)
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'f={scores["faithfulness"]:.2f} cr={scores["context_recall"]:.2f} '
              f'ar={scores["answer_relevancy"]:.2f} cp={scores["context_precision"]:.2f} | '
              f'avg: f={avg_f:.3f} cr={avg_cr:.3f} ar={avg_ar:.3f} cp={avg_cp:.3f} | ETA {eta:.1f}m')

print(f'\nSelesai! -> {PHASE2_CUSTOM_PATH}')

Mulai: 500 sampel (custom zero-NaN, 4 metrik).

Tahap 2: Evaluasi 500 sampel baru...

  [  5/500] idx=4 | f=1.00 cr=1.00 ar=1.00 cp=0.92 | avg: f=0.200 cr=0.633 ar=1.000 cp=0.967 | ETA 743.6m
  [ 10/500] idx=9 | f=0.33 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.400 cr=0.683 ar=1.000 cp=0.939 | ETA 711.9m
  [ 15/500] idx=14 | f=0.67 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.533 cr=0.789 ar=1.000 cp=0.960 | ETA 676.6m
  [ 20/500] idx=19 | f=0.40 cr=1.00 ar=1.00 cp=0.95 | avg: f=0.545 cr=0.842 ar=1.000 cp=0.967 | ETA 650.4m
  [ 25/500] idx=24 | f=0.67 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.569 cr=0.840 ar=1.000 cp=0.972 | ETA 637.6m
  [ 30/500] idx=29 | f=1.00 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.619 cr=0.833 ar=0.989 cp=0.973 | ETA 620.5m
  [ 35/500] idx=34 | f=0.50 cr=0.50 ar=1.00 cp=0.87 | avg: f=0.621 cr=0.814 ar=0.990 cp=0.971 | ETA 608.4m
  [ 40/500] idx=39 | f=1.00 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.643 cr=0.812 ar=0.992 cp=0.969 | ETA 593.9m
  [ 45/500] idx=44 | f=0.33 cr=1.00 ar=1.00 cp=1.00 | avg: f

## Summary — Hasil Akhir

In [17]:
with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
    p2 = json.load(f)['results']

n      = len(p2)
acc    = sum(r['is_correct']        for r in p2) / n
avg_f  = sum(r['faithfulness']      for r in p2) / n
avg_cr = sum(r['context_recall']    for r in p2) / n
avg_ar = sum(r['answer_relevancy']  for r in p2) / n
avg_cp = sum(r['context_precision'] for r in p2) / n

print('=' * 65)
print(f'  {CONFIG_NAME.upper()} (Hybrid Llama) - {n} sampel')
print(f'  LLM: {LLM_MODEL}')
print('=' * 65)
print(f'  Label Accuracy     : {acc:.1%}')
print(f'  Hallucination Rate : {1-acc:.1%}')
print(f'  Faithfulness       : {avg_f:.4f}')
print(f'  Context Recall     : {avg_cr:.4f}')
print(f'  Answer Relevancy   : {avg_ar:.4f}')
print(f'  Context Precision  : {avg_cp:.4f}')
print(f'  NaN count          : 0')
print('=' * 65)

print('\nPer-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = [r for r in p2 if r['ground_truth'] == lbl]
    if sub:
        lbl_acc = sum(r['is_correct'] for r in sub) / len(sub)
        print(f'  {lbl:>5}: {sum(r["is_correct"] for r in sub)}/{len(sub)} = {lbl_acc:.1%}')

# Perbandingan dengan konfigurasi OpenAI sebelumnya (benchmark reference)
print('\nPerbandingan dengan konfigurasi OpenAI sebagai benchmark:')
for prev_config, prev_path in [
    ('Baseline OpenAI', '../results/baseline_openai_phase2_custom.json'),
    ('QR OpenAI',       '../results/qr_openai_phase2_custom.json'),
    ('CR OpenAI',       '../results/cr_openai_phase2_custom.json'),
]:
    try:
        with open(prev_path, 'r', encoding='utf-8') as f:
            prev = json.load(f)['results']
        p_acc = sum(r['is_correct'] for r in prev) / len(prev)
        p_f   = sum(r['faithfulness'] for r in prev) / len(prev)
        delta = (acc - p_acc) * 100
        print(f'  {prev_config:<20}: acc={p_acc:.1%} faith={p_f:.4f} | delta acc = {delta:+.1f}%')
    except FileNotFoundError:
        print(f'  {prev_config:<20}: file tidak ditemukan')

print(f'\nBaris tabel skripsi:')
print(f'  | Hybrid Llama 3.3 70B | {acc:.3f} | {1-acc:.3f} | '
      f'{avg_f:.3f} | {avg_cr:.3f} | {avg_ar:.3f} | {avg_cp:.3f} |')

  QR_CR_LLAMA32 (Hybrid Llama) - 500 sampel
  LLM: llama3.2
  Label Accuracy     : 55.0%
  Hallucination Rate : 45.0%
  Faithfulness       : 0.6277
  Context Recall     : 0.8599
  Answer Relevancy   : 0.9714
  Context Precision  : 0.9559
  NaN count          : 0

Per-label accuracy:
    yes: 250/275 = 90.9%
     no: 23/159 = 14.5%
  maybe: 2/66 = 3.0%

Perbandingan dengan konfigurasi OpenAI sebagai benchmark:
  Baseline OpenAI     : acc=69.2% faith=0.8930 | delta acc = -14.2%
  QR OpenAI           : acc=70.8% faith=0.8967 | delta acc = -15.8%
  CR OpenAI           : acc=69.8% faith=0.8837 | delta acc = -14.8%

Baris tabel skripsi:
  | Hybrid Llama 3.3 70B | 0.550 | 0.450 | 0.628 | 0.860 | 0.971 | 0.956 |
